# Exact-match confirmation of CODI's accuracy-bearing PC band

## What is being confirmed

The analytic tier of the margin-geometry experiment found, on held-out **first-token**
accuracy at the forced answer cue:

| subspace | dims | % of variance | retained accuracy |
|---|---:|---:|---:|
| PC 0–3 | 4 | **82.3%** | **0.067** |
| PC 4–15 | 12 | 7.5% | 0.506 |
| **PC 4–31** | **28** | **11.3%** | **0.859** |
| PC 32–767 | 736 | 6.4% | 0.222 |

Variance rank and answer contribution are almost unrelated: the leading component
holds two thirds of all variance and 6% of the accuracy. This notebook re-tests that
with real greedy decoding and **numeric exact match**.

## Preregistered gates, frozen before any exact-match outcome

1. **Sufficiency** — retaining PC 4–31 preserves ≥ 70% of baseline accuracy.
2. **Dissociation** — retaining PC 0–3 preserves ≤ 20% of baseline, and the primary
   band beats it with a positive paired bootstrap lower bound.
3. **Necessity** — removing PC 4–31 costs ≥ 20 accuracy points, with a positive lower
   bound and exact McNemar p ≤ 0.05.

All three must pass. Random-subspace arms are descriptive; the specificity null was
already established analytically with 200 energy-matched replicates.

Precision is pinned to float32 for reproducibility, but note that precision is **not**
the cause of the baseline shift seen previously: with float32 explicitly resolved the
forced-cue baseline was 0.4056, against 0.4041 under `auto` — two answers apart. The
reproduction gate is therefore re-decoded on the current image before any arm runs,
and the baseline-drift guard references that fresh value.

No weight is updated and no speed claim is made. Enable Internet and a GPU, then
choose **Save Version → Save & Run All**.

## Setup

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the immutable commit after pushing.
REPO_DIR = "/kaggle/working/latent-reasoning"
REPRODUCTION_SUMMARY_INPUT = ""
ENERGY_BASIS_INPUT = ""
ANSWER_CONDITIONED_BASIS_INPUT = ""
PARAMETER_AWARE_BASIS_INPUT = ""
# Attach the completed margin-geometry export; its float32 colon states are reused.
COLON_STATES_INPUT = ""
READOUT_INPUT = ""
RESUME_INPUT = ""

EVAL_BATCH_SIZE = 32
GENERATION_PRECISION = "float32"
PRIMARY_BAND = (4, 32)
CONTROL_BAND = (0, 4)
BAND_RANDOM_REPLICATES = 4

import os, pathlib, subprocess, sys, json, glob

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip())

## Pin the environment that reproduces the checkpoint

The run that established the 43.669% GSM8K gate recorded its packages in the eval manifest: transformers 4.52.4, peft 0.15.2, datasets 3.6.0, huggingface_hub 0.32.4, torch 2.10.0+cu128. The current Kaggle image keeps the same torch but ships much newer transformers and peft, and on it the native gate scores **0.3723** instead of 0.4367.

`src/models/official_codi.py` states outright that its cache handling is written against Transformers 4.52 legacy-tuple semantics, which later versions changed. The CODI latent loop threads `past_key_values` through six hand-rolled forward passes, so a change there degrades it silently rather than raising.

The pins are installed before any script runs. Scripts execute as subprocesses and import fresh, so no kernel restart is needed.

In [ ]:
PINNED_PACKAGES = {
    "transformers": "4.52.4",
    "peft": "0.15.2",
    "datasets": "3.6.0",
    "huggingface_hub": "0.32.4",
}

import importlib.metadata as _md


def _installed(name):
    try:
        return _md.version(name)
    except _md.PackageNotFoundError:
        return None


before = {name: _installed(name) for name in PINNED_PACKAGES}
print("before:", before)
missing = [f"{n}=={v}" for n, v in PINNED_PACKAGES.items() if before.get(n) != v]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

probe = (
    "import importlib.metadata as m;"
    "print({n: m.version(n) for n in "
    f"{list(PINNED_PACKAGES)!r}" "})"
)
after = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
print("after :", after.stdout.strip() or after.stderr.strip())
for name, wanted in PINNED_PACKAGES.items():
    assert f"'{name}': '{wanted}'" in after.stdout, (name, wanted, after.stdout)
import torch as _torch
print("torch  :", _torch.__version__, "(unchanged from the reproducing run)")

## Repair the peft/torchao environment

`peft`'s LoRA dispatcher raises instead of returning `False` when torchao is installed below its minimum. torchao is optional here, so removing an incompatible copy restores the clean path. No-op when it is absent or current.

In [ ]:
def _peft_torchao_state():
    probe = (
        "from peft.import_utils import is_torchao_available\n"
        "try:\n"
        "    print('ok' if is_torchao_available() else 'absent')\n"
        "except ImportError as error:\n"
        "    print('incompatible:' + str(error))\n"
    )
    result = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    return (result.stdout + result.stderr).strip()


state = _peft_torchao_state()
print("before:", state)
if state.startswith("incompatible"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
    state = _peft_torchao_state()
    print("after:", state)
assert not state.startswith("incompatible"), f"peft still cannot dispatch LoRA: {state}"

## Source tests

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_endpoint_band_confirmation.py",
     "tests/test_endpoint_margin_geometry.py"],
    check=True,
)

## Resolve inputs

The colon states are reused from the completed margin-geometry run, so the band bases here are the same ones the analytic tier measured.

In [ ]:
OUTPUT_ROOT = pathlib.Path("/kaggle/working/latent-reasoning/outputs/official_codi_endpoint_band_confirmation")
REPORT_ROOT = pathlib.Path("/kaggle/working/latent-reasoning/reports/official_codi_endpoint_band_confirmation")
LOG_ROOT = pathlib.Path("/kaggle/working/latent-reasoning/logs/official_codi_endpoint_band_confirmation")
for path in (OUTPUT_ROOT, REPORT_ROOT, LOG_ROOT):
    path.mkdir(parents=True, exist_ok=True)

# Restore a previous session's outputs when one is attached. Every arm and the
# reproduction gate are keyed by request hash, so restored work is skipped and only
# the missing steps run.
if RESUME_INPUT:
    import shutil
    restored = 0
    for source in pathlib.Path(RESUME_INPUT).rglob("official_codi_endpoint_band_confirmation"):
        if not source.is_dir():
            continue
        for item in source.rglob("*"):
            if item.is_file():
                target = OUTPUT_ROOT / item.relative_to(source)
                if not target.exists():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(item, target)
                    restored += 1
    print("restored files from RESUME_INPUT:", restored)


def _discover(explicit, pattern):
    if explicit:
        return explicit
    matches = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert matches, f"attach a dataset containing {pattern}"
    return sorted(matches, key=lambda value: (len(value.split("/")), value))[0]


COLON_STATES = _discover(COLON_STATES_INPUT, "colon_states_seed89/colon_states.pt")
READOUT = _discover(READOUT_INPUT, "colon_states_seed89/readout.pt")
ENERGY_BASIS = _discover(ENERGY_BASIS_INPUT,
    "official_codi_endpoint_tsvc_corrected/calibration_seed11/basis.pt")
ANSWER_CONDITIONED_BASIS = _discover(ANSWER_CONDITIONED_BASIS_INPUT,
    "official_codi_endpoint_answer_conditioned/collection_seed29/basis.pt")
PARAMETER_AWARE_BASIS = _discover(PARAMETER_AWARE_BASIS_INPUT,
    "official_codi_endpoint_parameter_aware/collection_seed41/basis.pt")
REPRODUCTION_SUMMARY = _discover(REPRODUCTION_SUMMARY_INPUT,
    "official_codi_gpt2/eval/revision_fd641b3d/full_gsm8k/summary.json")

import torch
cache = torch.load(COLON_STATES, map_location="cpu", weights_only=False)
assert cache["parity_gate"]["passed"], cache["parity_gate"]
assert cache["metadata"]["precision"] == "float32", cache["metadata"]["precision"]
print("colon states:", COLON_STATES)
print("parity agreement:", cache["full_parity_gate"]["agreement"])

## Re-establish the reproduction gate on **this** environment

The attached reproduction summary was computed months ago on a different Kaggle image. It cannot detect the environment changing underneath it, and the previous run showed exactly that: with precision explicitly pinned to float32 the forced-cue baseline was 0.4056, against a historical 0.4329. Precision was ruled out as the cause.

So the gate is re-decoded here, natively, on the current image. Everything downstream references the fresh summary, and `verify_full_reproduction_gate` raises if it falls outside the preregistered 0.437 +/- 0.03 band. A failure here is itself the result: this environment would no longer reproduce the checkpoint, and no mechanistic arm run on it is comparable to the completed experiments.

In [ ]:
def run_persisted(command, log_name):
    log_path = LOG_ROOT / log_name
    with open(log_path, "w") as log:
        process = subprocess.Popen(command, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="", flush=True); log.write(line)
        code_ = process.wait()
    if code_ != 0:
        raise RuntimeError(f"Command failed with exit code {code_}; inspect {log_path}")
    return log_path


FRESH_GATE_ROOT = OUTPUT_ROOT / "reproduction_gate"
FRESH_REPRODUCTION_SUMMARY = (
    FRESH_GATE_ROOT / "eval" / "revision_fd641b3d" / "full_gsm8k" / "summary.json"
)
if not FRESH_REPRODUCTION_SUMMARY.is_file():
    run_persisted(
        [sys.executable, "-u", "-m", "src.eval.official_codi",
         "--config", "configs/official_codi_gpt2.yaml",
         "--datasets", "gsm8k", "--limit", "0",
         "--output-dir", str(FRESH_GATE_ROOT),
         "--device", "cuda"],
        "fresh_reproduction_gate.log",
    )
fresh = json.loads(FRESH_REPRODUCTION_SUMMARY.read_text())
attached = json.loads(pathlib.Path(REPRODUCTION_SUMMARY).read_text())


# One shared reader, so the notebook and the analysis CLI cannot disagree about
# the summary schema.
from src.eval.official_codi_endpoint_band_confirmation_analysis import (
    gsm8k_accuracy_from_summary,
)

fresh_acc = gsm8k_accuracy_from_summary(fresh)
attached_acc = gsm8k_accuracy_from_summary(attached)
print(f"native GSM8K accuracy on THIS image : {fresh_acc}")
print(f"native GSM8K accuracy when recorded : {attached_acc}")
print(f"drift: {abs(float(fresh_acc) - float(attached_acc)):.6f}")
if abs(float(fresh_acc) - float(attached_acc)) > 0.01:
    print()
    print("WARNING: this environment does not reproduce the recorded checkpoint "
          "accuracy. Results here are internally consistent but are NOT comparable "
          "to the completed experiments until the cause is identified.")
# Every downstream arm references the fresh gate, not the historical one.
REPRODUCTION_SUMMARY = str(FRESH_REPRODUCTION_SUMMARY)

## Report the band geometry the arms will test

Printed before decoding so the variance shares are on the record next to the accuracies they are about to be compared with.

In [ ]:
from src.mech.endpoint_margin_geometry import (
    ANALYTIC_STATE, band_variance_share, state_covariance,
)

si = list(cache["state_order"]).index(ANALYTIC_STATE)
calibration = cache["calibration_states"][:, si, :]
mean = cache["student_mean"][ANALYTIC_STATE]
covariance = state_covariance(calibration - mean.unsqueeze(0))
BANDS = [(0, 4), (4, 16), (4, 32), (0, 32), (32, 768)]
for start, stop in BANDS:
    print(f"PC[{start}:{stop}) {stop - start:4d} dims  "
          f"variance share {100 * band_variance_share(covariance, start, stop):6.2f}%")

## Run the confirmation arms

Twelve full-GSM8K greedy decodes at pinned float32. The baseline arm asserts its own accuracy against the reproduction gate, so a precision regression stops the run instead of silently shifting every comparison.

In [ ]:
def band_arm(start, stop):
    return f"band_p{start:03d}_{stop:03d}_s{ANALYTIC_STATE}"


ARMS = [("baseline", "remove")]
ARMS += [(band_arm(*b), "retain") for b in BANDS]
ARMS += [(band_arm(*PRIMARY_BAND), "remove"), (band_arm(*CONTROL_BAND), "remove")]
ARMS += [(f"random_matched_band_k{PRIMARY_BAND[1] - PRIMARY_BAND[0]:03d}"
          f"_s{ANALYTIC_STATE}_r{i:03d}", "retain") for i in range(BAND_RANDOM_REPLICATES)]
assert len(ARMS) == 12, len(ARMS)

RUNS_ROOT = OUTPUT_ROOT / "runs"
for arm, mode in ARMS:
    tag = f"{arm}_{mode}"
    run_persisted(
        [sys.executable, "-u", "scripts/run_official_codi_endpoint_margin_generation.py",
         "--config", "configs/official_codi_gpt2.yaml",
         "--reproduction-summary", REPRODUCTION_SUMMARY,
         "--states", COLON_STATES, "--readout", READOUT,
         "--energy-basis", ENERGY_BASIS,
         "--answer-conditioned-basis", ANSWER_CONDITIONED_BASIS,
         "--parameter-aware-basis", PARAMETER_AWARE_BASIS,
         "--output-dir", str(RUNS_ROOT / tag),
         "--arm", arm, "--state", str(ANALYTIC_STATE), "--mode", mode,
         "--semantics", "mean",
         "--eval-batch-size", str(EVAL_BATCH_SIZE),
         "--precision", GENERATION_PRECISION,
         "--device", "cuda"],
        f"{tag}.log",
    )
completed = sorted(RUNS_ROOT.rglob("summary.json"))
print("completed arms:", len(completed), "/", len(ARMS))
baseline = [json.loads(p.read_text()) for p in completed]
baseline = [s for s in baseline if s["arm"] == "baseline"][0]
print("baseline exact match:", baseline["accuracy"],
      "drift:", baseline.get("baseline_accuracy_drift"),
      "passed:", baseline.get("baseline_drift_passed"))
assert baseline.get("baseline_drift_passed") is True, baseline

## Apply the preregistered gates

In [ ]:
REPORT = REPORT_ROOT / "band_confirmation_summary.json"
run_persisted(
    [sys.executable, "-u", "scripts/analyze_official_codi_endpoint_band_confirmation.py",
     "--config", "configs/official_codi_gpt2.yaml",
     "--runs-root", str(RUNS_ROOT),
     "--reproduction-summary", REPRODUCTION_SUMMARY,
     "--output", str(REPORT)],
    "analyze_band_confirmation.log",
)
report = json.loads(REPORT.read_text())
print("status:", report["status"])
for name, gate in report["gates"].items():
    print(f"  {name}: passed={gate['passed']}")

## Checksummed export

In [ ]:
import hashlib, shutil

EXPORT = pathlib.Path("/kaggle/working/official_codi_endpoint_band_confirmation_export")
if EXPORT.exists():
    shutil.rmtree(EXPORT)
EXPORT.mkdir(parents=True)
for source in (OUTPUT_ROOT, REPORT_ROOT, LOG_ROOT):
    target = EXPORT / source.relative_to("/kaggle/working")
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, target)
lines = []
for path in sorted(EXPORT.rglob("*")):
    if path.is_file():
        lines.append(f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.relative_to(EXPORT)}")
(EXPORT / "SHA256SUMS.txt").write_text("\n".join(lines) + "\n")
print("exported files:", len(lines))